# 04 - Ensemble Fusion for Multi-Modal Dementia Prediction

This notebook demonstrates ensemble fusion techniques combining predictions from tabular models and CNN models.

---

## Outline
- Load Pre-trained Models
- Generate Predictions from Base Models
- Late Fusion with Stacking
- Voting Ensemble
- Ensemble Evaluation
- Performance Comparison

---

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score

# Add src to path
sys.path.append('../src')
from ensemble import LateFusionStacker

# Display settings
pd.set_option('display.max_columns', 100)
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Pre-trained Models

Load the models trained in previous notebooks.

In [ ]:
# Load tabular models
models = {}
model_dir = '../models'

try:
    with open(os.path.join(model_dir, 'logistic_regression.pkl'), 'rb') as f:
        models['Logistic Regression'] = pickle.load(f)
    
    with open(os.path.join(model_dir, 'random_forest.pkl'), 'rb') as f:
        models['Random Forest'] = pickle.load(f)
    
    with open(os.path.join(model_dir, 'gradient_boosting.pkl'), 'rb') as f:
        models['Gradient Boosting'] = pickle.load(f)
    
    # Load preprocessor
    with open(os.path.join(model_dir, 'preprocessor.pkl'), 'rb') as f:
        preprocessor = pickle.load(f)
    
    print("Loaded tabular models:")
    for name in models.keys():
        print(f"  - {name}")
    
except FileNotFoundError as e:
    print(f"Error loading models: {e}")
    print("Please run notebooks 02 first to train and save models.")
    models = None

## 2. Prepare Test Data

Load and preprocess test data to generate predictions.

In [ ]:
# Load test data
# NOTE: This is a simplified example. In practice, you would load the same test set used in notebook 02.
from data_loading import load_clinical_data
from sklearn.model_selection import train_test_split

if models:
    try:
        # Load clinical data
        clinical_path = '../data/raw/clinical.csv'
        df = load_clinical_data(clinical_path)
        
        # Define features and target (adjust based on your data)
        numeric_features = ['Age', 'EDUC', 'MMSE', 'eTIV', 'nWBV', 'ASF']
        categorical_features = ['M/F']
        target_column = 'CDR'
        
        # Clean data
        df_clean = df.dropna(subset=[target_column])
        X = df_clean[[col for col in numeric_features + categorical_features if col in df_clean.columns]]
        y = df_clean[target_column]
        y_binary = (y > 0).astype(int)
        
        # Split (using same random state as notebook 02)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
        )
        
        # Preprocess
        X_test_processed = preprocessor.transform(X_test)
        feature_names = preprocessor.get_feature_names_out()
        X_test_processed = pd.DataFrame(
            X_test_processed, columns=feature_names, index=X_test.index
        )
        
        print(f"Test set shape: {X_test_processed.shape}")
        print(f"Test labels: {y_test.value_counts().to_dict()}")
        
    except Exception as e:
        print(f"Error loading test data: {e}")
        X_test_processed = None
        y_test = None

## 3. Generate Base Model Predictions

Generate predictions from each base model to be used in ensemble.

In [ ]:
# Generate predictions
base_predictions = {}
base_probabilities = {}

if models and X_test_processed is not None:
    for name, model in models.items():
        y_pred = model.predict(X_test_processed)
        y_pred_proba = model.predict_proba(X_test_processed)
        
        base_predictions[name] = y_pred
        base_probabilities[name] = y_pred_proba
        
        # Calculate AUC
        auc = roc_auc_score(y_test, y_pred_proba[:, 1])
        acc = accuracy_score(y_test, y_pred)
        
        print(f"{name}: Accuracy={acc:.4f}, AUC={auc:.4f}")

## 4. Late Fusion with Stacking

Use stacking to combine base model predictions with a meta-learner.

In [ ]:
# Prepare training data for stacking (from notebook 02)
if models and X_test_processed is not None:
    # Preprocess training data
    X_train_processed = preprocessor.transform(X_train)
    X_train_processed = pd.DataFrame(
        X_train_processed, columns=feature_names, index=X_train.index
    )
    
    # Create base learners list for stacking
    base_learners = [
        ('lr', models['Logistic Regression']),
        ('rf', models['Random Forest']),
        ('gbm', models['Gradient Boosting'])
    ]
    
    # Create and train stacking ensemble
    meta_learner = LogisticRegression(max_iter=1000, random_state=42)
    stacking_model = LateFusionStacker(
        base_learners=base_learners,
        meta_learner=meta_learner,
        cv=5,
        n_jobs=-1
    )
    
    print("Training stacking ensemble...")
    stacking_model.fit(X_train_processed, y_train)
    
    # Predictions
    stacking_pred = stacking_model.predict(X_test_processed)
    stacking_proba = stacking_model.predict_proba(X_test_processed)
    
    # Evaluate
    stacking_acc = accuracy_score(y_test, stacking_pred)
    stacking_auc = roc_auc_score(y_test, stacking_proba[:, 1])
    
    print(f"\nStacking Ensemble: Accuracy={stacking_acc:.4f}, AUC={stacking_auc:.4f}")

## 5. Voting Ensemble

Create a simple voting ensemble for comparison.

In [ ]:
# Create voting ensemble
if models and X_test_processed is not None:
    voting_model = VotingClassifier(
        estimators=base_learners,
        voting='soft',
        n_jobs=-1
    )
    
    print("Training voting ensemble...")
    voting_model.fit(X_train_processed, y_train)
    
    # Predictions
    voting_pred = voting_model.predict(X_test_processed)
    voting_proba = voting_model.predict_proba(X_test_processed)
    
    # Evaluate
    voting_acc = accuracy_score(y_test, voting_pred)
    voting_auc = roc_auc_score(y_test, voting_proba[:, 1])
    
    print(f"Voting Ensemble: Accuracy={voting_acc:.4f}, AUC={voting_auc:.4f}")

## 6. Ensemble Performance Comparison

In [ ]:
# Compare all models and ensembles
if models and X_test_processed is not None:
    results = []
    
    # Base models
    for name in models.keys():
        acc = accuracy_score(y_test, base_predictions[name])
        auc = roc_auc_score(y_test, base_probabilities[name][:, 1])
        results.append({'Model': name, 'Accuracy': acc, 'AUC-ROC': auc, 'Type': 'Base Model'})
    
    # Ensemble models
    results.append({'Model': 'Stacking Ensemble', 'Accuracy': stacking_acc, 'AUC-ROC': stacking_auc, 'Type': 'Ensemble'})
    results.append({'Model': 'Voting Ensemble', 'Accuracy': voting_acc, 'AUC-ROC': voting_auc, 'Type': 'Ensemble'})
    
    # Create summary DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('AUC-ROC', ascending=False)
    
    print("\nPerformance Comparison:")
    display(results_df)
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy comparison
    colors = ['skyblue' if t == 'Base Model' else 'coral' for t in results_df['Type']]
    ax1.barh(results_df['Model'], results_df['Accuracy'], color=colors)
    ax1.set_xlabel('Accuracy')
    ax1.set_title('Model Accuracy Comparison')
    ax1.set_xlim([0, 1])
    ax1.grid(True, alpha=0.3)
    
    # AUC comparison
    ax2.barh(results_df['Model'], results_df['AUC-ROC'], color=colors)
    ax2.set_xlabel('AUC-ROC')
    ax2.set_title('Model AUC-ROC Comparison')
    ax2.set_xlim([0, 1])
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. ROC Curves Comparison

In [ ]:
# Plot ROC curves for all models
if models and X_test_processed is not None:
    plt.figure(figsize=(10, 8))
    
    # Base models
    for name in models.keys():
        fpr, tpr, _ = roc_curve(y_test, base_probabilities[name][:, 1])
        auc = roc_auc_score(y_test, base_probabilities[name][:, 1])
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linestyle='--', alpha=0.7)
    
    # Ensemble models
    fpr, tpr, _ = roc_curve(y_test, stacking_proba[:, 1])
    plt.plot(fpr, tpr, label=f"Stacking Ensemble (AUC={stacking_auc:.3f})", linewidth=2)
    
    fpr, tpr, _ = roc_curve(y_test, voting_proba[:, 1])
    plt.plot(fpr, tpr, label=f"Voting Ensemble (AUC={voting_auc:.3f})", linewidth=2)
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.3)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - All Models and Ensembles')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.show()

## 8. Detailed Evaluation of Best Model

In [ ]:
# Evaluate the best performing ensemble
if models and X_test_processed is not None:
    # Choose best model based on AUC
    best_idx = results_df['AUC-ROC'].idxmax()
    best_model_name = results_df.loc[best_idx, 'Model']
    
    print(f"Best Model: {best_model_name}\n")
    
    # Get predictions for best model
    if best_model_name == 'Stacking Ensemble':
        best_pred = stacking_pred
    elif best_model_name == 'Voting Ensemble':
        best_pred = voting_pred
    else:
        best_pred = base_predictions[best_model_name]
    
    # Classification report
    print("Classification Report:")
    print(classification_report(y_test, best_pred, target_names=['Non-Demented', 'Demented']))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, best_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Demented', 'Demented'], 
                yticklabels=['Non-Demented', 'Demented'])
    plt.title(f'Confusion Matrix - {best_model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

## 9. Save Ensemble Models

In [ ]:
# Save ensemble models
if models and X_test_processed is not None:
    os.makedirs('../models', exist_ok=True)
    
    # Save stacking ensemble
    with open('../models/stacking_ensemble.pkl', 'wb') as f:
        pickle.dump(stacking_model, f)
    print("Saved stacking ensemble to ../models/stacking_ensemble.pkl")
    
    # Save voting ensemble
    with open('../models/voting_ensemble.pkl', 'wb') as f:
        pickle.dump(voting_model, f)
    print("Saved voting ensemble to ../models/voting_ensemble.pkl")
    
    # Save results
    os.makedirs('../outputs', exist_ok=True)
    results_df.to_csv('../outputs/ensemble_results.csv', index=False)
    print("Saved results to ../outputs/ensemble_results.csv")

## Summary

This notebook demonstrated:
- Loading pre-trained base models from previous notebooks
- Creating ensemble models using stacking and voting
- Comparing performance of base models vs. ensembles
- Visualizing ROC curves and performance metrics
- Identifying and evaluating the best performing model
- Saving ensemble models for future use

### Key Findings
- Ensemble methods typically improve upon individual base models
- Stacking allows the meta-learner to learn optimal combinations
- Voting provides a simpler alternative with competitive performance

### Next Steps
- Apply explainability techniques to understand ensemble predictions (notebook 05)
- Generate publication-ready results and figures (notebook 06)